# 📊 المشروع المشترك: المرحلة الثانية - التنبؤ الزمني والتنفيذ بـ Python
### إعداد: مجموعة التنبؤ المالي
المطلوب: أخذ الملف النظيف وتوليد توقعات دقيقة للـ 90 يوماً القادمة باستخدام مكتبة Prophet مع مراعاة تأثير الإجازات.

In [ ]:
# 1. تثبيت واستدعاء المكتبات المطلوبة
!pip install prophet matplotlib pandas

import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet

In [ ]:
# 2. قراءة بيانات المبيعات المجمعة من مرحلة الـ SQL وفحصها
df = pd.read_csv('daily_sales_cleaned.csv')
df['date'] = pd.to_datetime(df['date'])

print("--- أول 5 صفوف من البيانات المرفوعة ---")
print(df.head())

In [ ]:
# 3. Prophet إعداد جداول البيانات الخاصة بـ
holidays_df = df[df['is_holiday'] == 1][['date']].copy()
holidays_df = holidays_df.rename(columns={'date': 'ds'})
holidays_df['holiday'] = 'official_holiday'

df_prophet = df.rename(columns={'date': 'ds', 'total_sales': 'y'})[['ds', 'y']]

In [ ]:
# 4. بناء وتدريب نموذج التنبؤ وتوليد توقعات الـ 90 يوماً القادمة
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    holidays=holidays_df
)
model.fit(df_prophet)

future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

In [ ]:
# 5. رسم المخططات البيانية مراجعة النتائج
fig1 = model.plot(forecast)
plt.title("Store Sales Forecast - Next 90 Days", fontsize=14)
plt.xlabel("Date")
plt.ylabel("Total Sales")
plt.show()

fig2 = model.plot_components(forecast)
plt.show()

In [ ]:
# 6. تصدير النتائج وتجهيزها لمرحلة الـ Power BI
forecast_result = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
forecast_result = forecast_result.rename(columns={'ds': 'date'})
forecast_result.to_csv('forecast_output.csv', index=False)

print("✅ تم تصدير ملف (forecast_output.csv) بنجاح!")